# mmraz-qwen-probe-variations-question-options-answer-vast

Goal: replicate the core `mmraz-probe-variations-question-options-answer.ipynb` workflow on **Qwen/Qwen3-4B** for Vast runs, while restricting probe training/evaluation to layers **10, 14, 18, 22, 26**.

This notebook:
- uses the `Question + Options + Answer:` prompt family
- extracts both **mean answer-token** and **last answer-token** residual features
- trains four probe families: **LR**, **WLR**, **MM**, **WMM**
- fits three training regimes:
  - `explicit_train_only`
  - `explicit_train_plus_redteam_20260322-090107`
  - `explicit_train_plus_redteam_20260322-090107_plus_20260403-183445`
- saves a single artifact bundle containing **all trained probes** across regimes, features, and selected layers
- saves metrics, summaries, and comparison figures under `results/qwen_question_options_answer_probe_variations_vast/`

The downstream time-utility notebook is expected to consume the **explicit-train-only / mean-answer-tokens / MM steering vector** from the saved artifact bundle.


In [ ]:
import os
import gc
import hashlib
import json
import re
import shutil
import subprocess
import time
import warnings
from contextlib import nullcontext
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

os.environ.setdefault('MPLCONFIGDIR', str(Path('/tmp/mplconfig')))

try:
    plt.style.use('seaborn-v0_8-whitegrid')
except OSError:
    pass

pd.set_option('display.max_rows', 200)
pd.set_option('display.max_columns', 200)


In [ ]:
def find_repo_root(start: Path) -> Path:
    p = start.resolve()
    for _ in range(8):
        if (p / 'pyproject.toml').exists() and (p / 'data').exists():
            return p
        p = p.parent
    raise RuntimeError('Could not locate repo root from current working directory.')


def pick_first_existing(paths):
    for p in paths:
        if p.exists():
            return p
    raise FileNotFoundError('None of these paths exist: ' + str(paths))


def load_pairs(path: Path):
    data = json.loads(path.read_text())
    if isinstance(data, dict) and 'pairs' in data:
        return data.get('metadata', {}), data['pairs']
    return {}, data


def sha256(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()


ROOT = find_repo_root(Path.cwd())
OUTPUT_ROOT = ROOT / 'results' / 'qwen_question_options_answer_probe_variations_vast'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
RUN_ID = time.strftime('%Y%m%d-%H%M%S')
SAVE_DIR = OUTPUT_ROOT / RUN_ID
SAVE_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = 'Qwen/Qwen3-4B'
USE_CHAT_TEMPLATE = True
DISABLE_THINKING_TRACE = True
SELECTED_LAYERS = [10, 14, 18, 22, 26]
FEATURE_SPECS = [
    {'name': 'mean_answer_tokens', 'pooling': 'mean'},
    {'name': 'last_answer_token', 'pooling': 'last'},
]
PAIR_SPLIT_RANDOM_STATE = 42
STRIP_OPTION_LETTERS_FOR_PROBE_TRAINING = True
NORMALIZE_STEERING_VECTORS = True
PATCH_PROMPT_LAST_ONLY = True
PATCH_GENERATION_TOKENS = True
PROBE_BATCH_SIZE = 1
WHITEN_REG = 1e-2
GPU_MONITOR_INTERVAL = 25
GPU_UTIL_WARN_THRESHOLD = 20
GPU_UNDERUTILIZED_STREAK_WARN = 3
PRIMARY_RED_TEAM_RUN_ID = '20260322-090107'
LATEST_RED_TEAM_RUN_ID = '20260403-183445'
TWO_RED_TEAM_RUNS_SLUG = f'{PRIMARY_RED_TEAM_RUN_ID}_plus_{LATEST_RED_TEAM_RUN_ID}'

explicit_expanded_path = pick_first_existing([
    ROOT / 'data/raw/temporal_scope_AB_randomized/temporal_scope_explicit_expanded_500.json',
    ROOT / 'data/raw/temporal_scope/temporal_scope_explicit_expanded_500.json',
    ROOT / 'data/raw/temporal_scope_explicit_expanded_500.json',
])
implicit_expanded_path = pick_first_existing([
    ROOT / 'data/raw/temporal_scope_AB_randomized/temporal_scope_implicit_expanded_300.json',
    ROOT / 'data/raw/temporal_scope/temporal_scope_implicit_expanded_300.json',
    ROOT / 'data/raw/temporal_scope_implicit_expanded_300.json',
])

exp_meta, explicit_pairs = load_pairs(explicit_expanded_path)
imp_meta, implicit_pairs = load_pairs(implicit_expanded_path)

print('Repo root:', ROOT)
print('Save dir:', SAVE_DIR)
print('Expanded explicit dataset:', explicit_expanded_path)
print('Expanded implicit dataset:', implicit_expanded_path)
print('Expanded explicit metadata:', exp_meta)
print('Expanded implicit metadata:', imp_meta)
print('Expanded explicit pairs:', len(explicit_pairs))
print('Expanded implicit pairs:', len(implicit_pairs))

dataset_hash_df = pd.DataFrame([
    {
        'dataset': 'explicit_expanded_500',
        'path': str(explicit_expanded_path.relative_to(ROOT)),
        'sha256': sha256(explicit_expanded_path),
    },
    {
        'dataset': 'implicit_expanded_300',
        'path': str(implicit_expanded_path.relative_to(ROOT)),
        'sha256': sha256(implicit_expanded_path),
    },
])
display(dataset_hash_df)


In [ ]:
def extract_option_letter(option_text):
    match = re.search(r'\(([ABab])\)', option_text or '')
    return match.group(1).upper() if match else None


def strip_option_label(option_text):
    return re.sub(r'^\s*\([ABab]\)\s*', '', option_text or '').strip()


def get_pair_option_payload(pair):
    immediate_letter = extract_option_letter(pair['immediate'])
    long_term_letter = extract_option_letter(pair['long_term'])
    if not immediate_letter or not long_term_letter or immediate_letter == long_term_letter:
        raise ValueError(f'Could not resolve A/B option ordering for pair: {pair!r}')

    option_a_text = pair['immediate'] if immediate_letter == 'A' else pair['long_term']
    option_b_text = pair['immediate'] if immediate_letter == 'B' else pair['long_term']

    return {
        'option_a_text': option_a_text,
        'option_b_text': option_b_text,
        'candidate_immediate_text': strip_option_label(pair['immediate']),
        'candidate_long_term_text': strip_option_label(pair['long_term']),
    }


def build_probe_prompt(question_text, option_a_text, option_b_text):
    question_text = (question_text or '').strip()
    option_a_text = strip_option_label(option_a_text)
    option_b_text = strip_option_label(option_b_text)
    return (
        f'{question_text}\n'
        'Options:\n'
        f'  {option_a_text}\n'
        f'  {option_b_text}\n'
        '  Answer:\n'
    )


def build_teacher_forced_examples_from_pairs(pairs, strip_option_letters=True):
    examples = []
    labels = []
    for pair_idx, pair in enumerate(pairs):
        option_payload = get_pair_option_payload(pair)
        prompt = build_probe_prompt(
            pair['question'],
            option_payload['option_a_text'],
            option_payload['option_b_text'],
        )
        immediate_continuation = option_payload['candidate_immediate_text']
        long_term_continuation = option_payload['candidate_long_term_text']
        if not strip_option_letters:
            immediate_continuation = pair['immediate']
            long_term_continuation = pair['long_term']

        examples.append({'prompt': prompt, 'continuation': immediate_continuation, 'label': 0, 'pair_idx': int(pair_idx)})
        labels.append(0)
        examples.append({'prompt': prompt, 'continuation': long_term_continuation, 'label': 1, 'pair_idx': int(pair_idx)})
        labels.append(1)
    return examples, np.array(labels, dtype=np.int64)


def load_stripped_red_team_examples(dataset_path):
    if not dataset_path.exists():
        raise FileNotFoundError(
            'Missing stripped red-team dataset: ' 
            + str(dataset_path)
            + '. Run scripts/mmraz_intertemporal/build_stripped_red_team_probe_dataset.py first.'
        )

    examples = []
    labels = []
    rows = []
    for line in dataset_path.read_text(encoding='utf-8').splitlines():
        if not line.strip():
            continue
        payload = json.loads(line)
        label_text = payload['judge_label']
        if label_text == 'short_term':
            label = 0
        elif label_text == 'long_term':
            label = 1
        else:
            raise ValueError(f'Unexpected judge_label: {label_text!r}')

        if all(key in payload for key in ['question_text', 'option_a_text', 'option_b_text']):
            probe_prompt = build_probe_prompt(
                payload['question_text'],
                payload['option_a_text'],
                payload['option_b_text'],
            )
        else:
            probe_prompt = payload['probe_prompt']

        probe_completion = payload.get('probe_completion') or payload.get('chosen_option_text')
        if not probe_prompt or not probe_completion:
            raise ValueError('Red-team example is missing probe_prompt/probe_completion fields.')

        examples.append({'prompt': probe_prompt, 'continuation': probe_completion, 'label': label})
        labels.append(label)
        rows.append({
            'candidate_id': payload.get('candidate_id'),
            'round_idx': payload.get('round_idx'),
            'judge_label': payload.get('judge_label'),
            'probe_label': payload.get('probe_label'),
            'is_adversarial_success': payload.get('is_adversarial_success'),
        })

    return examples, np.array(labels, dtype=np.int64), pd.DataFrame(rows)


def load_red_team_examples_for_run(run_id):
    run_dir = ROOT / 'out/mmraz_intertemporal/adversarial_red_teaming/runs' / run_id
    stripped_path = run_dir / 'probe_dataset_stripped.jsonl'
    if stripped_path.exists():
        examples, labels, rows = load_stripped_red_team_examples(stripped_path)
        return examples, labels, rows, stripped_path

    raw_candidates_path = run_dir / 'all_candidates.jsonl'
    if not raw_candidates_path.exists():
        raise FileNotFoundError(
            f'Missing both probe_dataset_stripped.jsonl and all_candidates.jsonl for red-team run {run_id}.'
        )

    examples = []
    labels = []
    rows = []
    for line in raw_candidates_path.read_text(encoding='utf-8').splitlines():
        if not line.strip():
            continue
        payload = json.loads(line)
        label_text = payload['judge_label']
        if label_text == 'short_term':
            label = 0
        elif label_text == 'long_term':
            label = 1
        else:
            raise ValueError(f'Unexpected judge_label: {label_text!r}')

        probe_prompt = payload.get('probe_prompt')
        if not probe_prompt and all(key in payload for key in ['question_text', 'option_a_text', 'option_b_text']):
            probe_prompt = build_probe_prompt(
                payload['question_text'],
                payload['option_a_text'],
                payload['option_b_text'],
            )
        probe_completion = payload.get('probe_completion') or payload.get('chosen_option_text')
        if not probe_prompt or not probe_completion:
            raise ValueError(
                f'Run {run_id} is missing probe_prompt/probe_completion fields needed for probe retraining.'
            )

        examples.append({'prompt': probe_prompt, 'continuation': probe_completion, 'label': label})
        labels.append(label)
        rows.append({
            'candidate_id': payload.get('candidate_id'),
            'round_idx': payload.get('round_idx'),
            'judge_label': payload.get('judge_label'),
            'probe_label': payload.get('probe_label'),
            'is_adversarial_success': payload.get('is_adversarial_success'),
        })

    return examples, np.array(labels, dtype=np.int64), pd.DataFrame(rows), raw_candidates_path


explicit_examples, y_exp = build_teacher_forced_examples_from_pairs(
    explicit_pairs,
    strip_option_letters=STRIP_OPTION_LETTERS_FOR_PROBE_TRAINING,
)
implicit_examples, y_imp = build_teacher_forced_examples_from_pairs(
    implicit_pairs,
    strip_option_letters=STRIP_OPTION_LETTERS_FOR_PROBE_TRAINING,
)

print('Expanded explicit samples:', len(y_exp), '| class balance:', np.bincount(y_exp))
print('Expanded implicit samples:', len(y_imp), '| class balance:', np.bincount(y_imp))
print('Prompt example:')
print(repr(explicit_examples[0]['prompt']))
print('Continuation example:')
print(repr(explicit_examples[0]['continuation']))

primary_red_team_examples, y_primary_red_team, primary_red_team_df, primary_red_team_path = load_red_team_examples_for_run(PRIMARY_RED_TEAM_RUN_ID)
latest_red_team_examples, y_latest_red_team, latest_red_team_df, latest_red_team_path = load_red_team_examples_for_run(LATEST_RED_TEAM_RUN_ID)

print('Primary red-team run:', PRIMARY_RED_TEAM_RUN_ID, '| path =', primary_red_team_path, '| examples =', len(y_primary_red_team))
print('Latest red-team run:', LATEST_RED_TEAM_RUN_ID, '| path =', latest_red_team_path, '| examples =', len(y_latest_red_team))
display(primary_red_team_df.groupby(['judge_label', 'is_adversarial_success']).size().rename('n_examples').reset_index())
display(latest_red_team_df.groupby(['judge_label', 'is_adversarial_success']).size().rename('n_examples').reset_index())


In [ ]:
qwen_device = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
print('Qwen device:', qwen_device)

NVIDIA_SMI_PATH = shutil.which('nvidia-smi')
if qwen_device == 'cuda' and NVIDIA_SMI_PATH is None:
    warnings.warn('nvidia-smi is not available; live GPU utilization monitoring will be unavailable.')


WORKING_CUDA_INDICES = []
PREFERRED_CUDA_INDEX = None


def discover_working_cuda_indices():
    if qwen_device != 'cuda':
        return [], {}
    try:
        torch.cuda.init()
    except Exception as exc:
        raise RuntimeError(
            'torch.cuda.is_available() was True, but CUDA initialization failed before model load. '
            f'Original error: {exc}'
        ) from exc

    working = []
    errors = {}
    for idx in range(torch.cuda.device_count()):
        try:
            device_name = torch.cuda.get_device_name(idx)
            probe = torch.empty(1, device=f'cuda:{idx}')
            del probe
            torch.cuda.synchronize(idx)
            working.append(idx)
            print(f'CUDA device {idx} ok: {device_name}')
        except Exception as exc:
            errors[idx] = str(exc)
            print(f'CUDA device {idx} unavailable: {exc}')
    return working, errors


def ensure_cuda_connection():
    global WORKING_CUDA_INDICES, PREFERRED_CUDA_INDEX
    if qwen_device != 'cuda':
        WORKING_CUDA_INDICES = []
        PREFERRED_CUDA_INDEX = None
        return []

    working, errors = discover_working_cuda_indices()
    WORKING_CUDA_INDICES = working
    PREFERRED_CUDA_INDEX = working[0] if working else None
    if not working:
        raise RuntimeError(
            'torch.cuda.is_available() was True, but no visible CUDA device accepted a test allocation. '
            f'CUDA_VISIBLE_DEVICES={os.environ.get("CUDA_VISIBLE_DEVICES")} | errors={errors}'
        )
    print(f'Using preferred CUDA device {PREFERRED_CUDA_INDEX}')
    return working


def _parse_cuda_index(value):
    text = str(value)
    if text.isdigit():
        return int(text)
    if text.startswith('cuda:'):
        return int(text.split(':', 1)[1])
    return None


def get_active_cuda_indices():
    maybe_model = globals().get('model', None)
    hf_device_map = getattr(maybe_model, 'hf_device_map', None)
    indices = []
    if hf_device_map is not None:
        for value in hf_device_map.values():
            idx = _parse_cuda_index(value)
            if idx is not None:
                indices.append(idx)
    if indices:
        return sorted(set(indices))
    if WORKING_CUDA_INDICES:
        return list(WORKING_CUDA_INDICES)
    return []


def get_gpu_status_rows():
    if qwen_device != 'cuda' or NVIDIA_SMI_PATH is None:
        return []
    result = subprocess.run(
        [
            NVIDIA_SMI_PATH,
            '--query-gpu=index,name,utilization.gpu,memory.used,memory.total',
            '--format=csv,noheader,nounits',
        ],
        capture_output=True,
        text=True,
        check=False,
    )
    if result.returncode != 0:
        warnings.warn(f'nvidia-smi failed with exit code {result.returncode}: {result.stderr.strip()}')
        return []

    active_indices = set(get_active_cuda_indices())
    rows = []
    for line in result.stdout.splitlines():
        if not line.strip():
            continue
        parts = [part.strip() for part in line.split(',')]
        if len(parts) != 5:
            continue
        gpu_index = int(parts[0])
        if active_indices and gpu_index not in active_indices:
            continue
        rows.append({
            'index': gpu_index,
            'name': parts[1],
            'util_pct': int(parts[2]),
            'memory_used_gib': float(parts[3]) / 1024.0,
            'memory_total_gib': float(parts[4]) / 1024.0,
        })
    return rows


def format_gpu_status(rows):
    if not rows:
        return 'gpu stats unavailable'
    return ' | '.join(
        f"gpu{row['index']} util={row['util_pct']}% mem={row['memory_used_gib']:.1f}/{row['memory_total_gib']:.1f} GiB"
        for row in rows
    )


def report_gpu_status(prefix):
    rows = get_gpu_status_rows()
    print(f'{prefix} | {format_gpu_status(rows)}')
    return rows


def looks_underutilized(rows):
    return bool(rows) and all(row['util_pct'] < GPU_UTIL_WARN_THRESHOLD for row in rows)


if qwen_device == 'cuda':
    ensure_cuda_connection()

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model_load_kwargs = {'trust_remote_code': True}
if qwen_device == 'cuda':
    model_load_kwargs['torch_dtype'] = torch.float16
    model_load_kwargs['device_map'] = {'': PREFERRED_CUDA_INDEX}
    print(f'Loading model on preferred CUDA device {PREFERRED_CUDA_INDEX}')
elif qwen_device == 'mps':
    model_load_kwargs['torch_dtype'] = torch.float16
else:
    model_load_kwargs['torch_dtype'] = torch.float32

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, **model_load_kwargs)
if qwen_device != 'cuda':
    model = model.to(qwen_device)
model.eval()

hf_device_map = getattr(model, 'hf_device_map', None)
MODEL_HAS_CPU_OFFLOAD = False
if hf_device_map is not None:
    device_counts = {}
    for value in hf_device_map.values():
        key = str(value)
        device_counts[key] = device_counts.get(key, 0) + 1
    print('hf_device_map device counts:', device_counts)
    offloaded_modules = [name for name, value in hf_device_map.items() if str(value) in {'cpu', 'disk'}]
    MODEL_HAS_CPU_OFFLOAD = bool(offloaded_modules)
    if MODEL_HAS_CPU_OFFLOAD:
        warnings.warn(
            f'Model is CPU/disk offloaded for {len(offloaded_modules)} modules. '
            f'Preview: {offloaded_modules[:8]}'
        )
else:
    print('No hf_device_map found; assuming single-device placement.')

ACTIVE_CUDA_INDICES = get_active_cuda_indices()
if qwen_device == 'cuda':
    print('Active CUDA indices:', ACTIVE_CUDA_INDICES)
    report_gpu_status('after model load')

n_layers = len(model.model.layers)
hidden_size = int(model.config.hidden_size)
invalid_layers = [layer for layer in SELECTED_LAYERS if layer < 0 or layer >= n_layers]
if invalid_layers:
    raise ValueError(f'SELECTED_LAYERS contains invalid Qwen layers: {invalid_layers}; n_layers={n_layers}')

print('Loaded', MODEL_NAME, '| n_layers =', n_layers, '| hidden_size =', hidden_size)
print('Selected layers:', SELECTED_LAYERS)


def get_model_device():
    if PREFERRED_CUDA_INDEX is not None:
        return torch.device(f'cuda:{PREFERRED_CUDA_INDEX}')
    active_cuda_indices = get_active_cuda_indices()
    if active_cuda_indices:
        return torch.device(f'cuda:{active_cuda_indices[0]}')
    return next(model.parameters()).device


def move_batch_to_model_device(batch):
    model_device = get_model_device()
    return {key: value.to(model_device) for key, value in batch.items()}


def clear_gpu_cache():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def format_prompt_for_model(user_prompt):
    if USE_CHAT_TEMPLATE:
        if not hasattr(tokenizer, 'apply_chat_template'):
            raise RuntimeError('Tokenizer does not expose apply_chat_template, but USE_CHAT_TEMPLATE=True was requested.')
        messages = [{'role': 'user', 'content': user_prompt}]
        if DISABLE_THINKING_TRACE:
            try:
                return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
            except TypeError:
                templated = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
                return templated + "<think>\n</think>\n\n"
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return user_prompt


def extract_answer_token_activations_qwen(examples, batch_size=1, dataset_name='dataset'):
    activations = {
        feature_spec['name']: {layer: [] for layer in SELECTED_LAYERS}
        for feature_spec in FEATURE_SPECS
    }
    pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id
    model_device = get_model_device()
    autocast_ctx = torch.autocast(device_type='cuda', dtype=torch.float16) if model_device.type == 'cuda' else nullcontext()
    underutilized_streak = 0
    start_time = time.time()

    progress = tqdm(
        range(0, len(examples), batch_size),
        total=(len(examples) + batch_size - 1) // batch_size,
        desc=f'Extracting {dataset_name}',
        unit='batch',
    )

    for start in progress:
        batch_examples = examples[start:start + batch_size]
        prompt_ids_batch = []
        full_ids_batch = []
        seq_lengths = []
        answer_spans = []

        for example in batch_examples:
            model_prompt = format_prompt_for_model(example['prompt'])
            prompt_ids = tokenizer(model_prompt, add_special_tokens=False, return_tensors='pt')['input_ids'][0]
            full_ids = tokenizer(model_prompt + example['continuation'], add_special_tokens=False, return_tensors='pt')['input_ids'][0]
            continuation_token_count = int(full_ids.shape[0] - prompt_ids.shape[0])
            if continuation_token_count <= 0:
                raise ValueError(f'Empty continuation for training example: {example!r}')
            prompt_ids_batch.append(prompt_ids)
            full_ids_batch.append(full_ids)
            seq_lengths.append(int(full_ids.shape[0]))

        max_seq_len = max(seq_lengths)
        input_ids = torch.full((len(batch_examples), max_seq_len), pad_id, dtype=torch.long)
        attention_mask = torch.zeros((len(batch_examples), max_seq_len), dtype=torch.long)

        for row_idx, (prompt_ids, seq) in enumerate(zip(prompt_ids_batch, full_ids_batch)):
            seq_len = int(seq.shape[0])
            answer_start = int(prompt_ids.shape[0])
            answer_end = seq_len
            input_ids[row_idx, :seq_len] = seq
            attention_mask[row_idx, :seq_len] = 1
            answer_spans.append((answer_start, answer_end))

        batch = move_batch_to_model_device({'input_ids': input_ids, 'attention_mask': attention_mask})

        with torch.inference_mode():
            with autocast_ctx:
                outputs = model(**batch, output_hidden_states=True, use_cache=False)

        for layer in SELECTED_LAYERS:
            hidden = outputs.hidden_states[layer + 1]
            pooled_mean_rows = []
            pooled_last_rows = []
            for row_idx, (answer_start, answer_end) in enumerate(answer_spans):
                answer_hidden = hidden[row_idx, answer_start:answer_end, :]
                pooled_mean_rows.append(answer_hidden.mean(dim=0).detach().float().cpu().numpy())
                pooled_last_rows.append(answer_hidden[-1, :].detach().float().cpu().numpy())
            activations['mean_answer_tokens'][layer].append(np.stack(pooled_mean_rows, axis=0))
            activations['last_answer_token'][layer].append(np.stack(pooled_last_rows, axis=0))

        del outputs
        clear_gpu_cache()

        completed = min(start + len(batch_examples), len(examples))
        if completed % GPU_MONITOR_INTERVAL == 0 or completed == len(examples):
            elapsed_min = (time.time() - start_time) / 60.0
            rows = get_gpu_status_rows()
            status_text = f'{dataset_name}: {completed}/{len(examples)} | elapsed {elapsed_min:.1f} min | {format_gpu_status(rows)}'
            progress.set_postfix_str(status_text.split(' | ', 1)[1] if ' | ' in status_text else status_text)
            print(status_text)
            if looks_underutilized(rows):
                underutilized_streak += 1
            else:
                underutilized_streak = 0
            if underutilized_streak >= GPU_UNDERUTILIZED_STREAK_WARN:
                warning_text = (
                    f'GPU looks underutilized for {dataset_name}: '
                    f'{underutilized_streak} consecutive checks below {GPU_UTIL_WARN_THRESHOLD}% util. '
                    f'Input device = {get_model_device()}. '
                )
                if MODEL_HAS_CPU_OFFLOAD:
                    warning_text += 'hf_device_map includes CPU/disk modules, so CPU offload is likely slowing inference.'
                else:
                    warning_text += 'If this persists, check nvidia-smi and confirm the active CUDA device is healthy.'
                warnings.warn(warning_text)
                underutilized_streak = 0

    for feature_name in activations:
        for layer in SELECTED_LAYERS:
            activations[feature_name][layer] = np.concatenate(activations[feature_name][layer], axis=0).astype(np.float32)

    return activations


In [ ]:
def normalize_direction(direction):
    norm = float(np.linalg.norm(direction))
    if norm <= 0:
        return direction.astype(np.float32), norm
    return (direction / norm).astype(np.float32), norm


def train_mm_probe(X_train, y_train):
    mu0 = X_train[y_train == 0].mean(axis=0)
    mu1 = X_train[y_train == 1].mean(axis=0)
    return (mu1 - mu0).astype(np.float32)


def mm_predict(X, direction):
    scores = X @ direction
    return (scores > 0).astype(np.int64), scores


def fit_whitener(X_train, reg=1e-2):
    X_train = X_train.astype(np.float64, copy=False)
    mean_train = X_train.mean(axis=0)
    Xc = X_train - mean_train

    cov = np.cov(Xc, rowvar=False, bias=False)
    avg_var = float(np.trace(cov) / cov.shape[0]) if cov.shape[0] > 0 else 1.0
    cov_reg = cov + (reg * avg_var) * np.eye(cov.shape[0], dtype=cov.dtype)

    precision = np.linalg.pinv(cov_reg)
    eigvals, eigvecs = np.linalg.eigh(cov_reg)
    eigvals = np.clip(eigvals, 1e-12, None)
    inv_sqrt = eigvecs @ np.diag(1.0 / np.sqrt(eigvals)) @ eigvecs.T

    try:
        cond = float(np.linalg.cond(cov_reg))
    except Exception:
        cond = float('nan')

    return {
        'mean_train': mean_train.astype(np.float32),
        'precision': precision.astype(np.float32),
        'inv_sqrt': inv_sqrt.astype(np.float32),
        'reg': float(reg),
        'cov_reg_condition_number': cond,
    }


def apply_whitener(X, whitener):
    Xc = X.astype(np.float32, copy=False) - whitener['mean_train']
    return Xc @ whitener['inv_sqrt']


def train_whitened_mm_probe(X_train, y_train, reg=1e-2):
    mu0 = X_train[y_train == 0].mean(axis=0)
    mu1 = X_train[y_train == 1].mean(axis=0)
    mm_direction = (mu1 - mu0).astype(np.float32)
    whitener = fit_whitener(X_train, reg=reg)
    effective_direction = (whitener['precision'] @ mm_direction).astype(np.float32)
    return {
        'mean_train': whitener['mean_train'],
        'effective_direction': effective_direction,
        'reg': whitener['reg'],
        'cov_reg_condition_number': whitener['cov_reg_condition_number'],
    }


def whitened_mm_predict(X, model):
    Xc = X - model['mean_train']
    scores = Xc @ model['effective_direction']
    return (scores > 0).astype(np.int64), scores


def summarize_probe_metrics(metrics_df):
    summary_rows = []
    for train_dataset, dataset_df in metrics_df.groupby('train_dataset'):
        for feature_name, feature_df in dataset_df.groupby('feature_name'):
            summary_rows.append({
                'train_dataset': train_dataset,
                'feature_name': feature_name,
                'best_lr_explicit_layer': int(feature_df.loc[feature_df['lr_explicit_test_acc'].idxmax(), 'layer']),
                'best_lr_explicit_acc': float(feature_df['lr_explicit_test_acc'].max()),
                'best_lr_implicit_layer': int(feature_df.loc[feature_df['lr_implicit_acc'].idxmax(), 'layer']),
                'best_lr_implicit_acc': float(feature_df['lr_implicit_acc'].max()),
                'best_wlr_explicit_layer': int(feature_df.loc[feature_df['wlr_explicit_test_acc'].idxmax(), 'layer']),
                'best_wlr_explicit_acc': float(feature_df['wlr_explicit_test_acc'].max()),
                'best_wlr_implicit_layer': int(feature_df.loc[feature_df['wlr_implicit_acc'].idxmax(), 'layer']),
                'best_wlr_implicit_acc': float(feature_df['wlr_implicit_acc'].max()),
                'best_mm_explicit_layer': int(feature_df.loc[feature_df['mm_explicit_test_acc'].idxmax(), 'layer']),
                'best_mm_explicit_acc': float(feature_df['mm_explicit_test_acc'].max()),
                'best_mm_implicit_layer': int(feature_df.loc[feature_df['mm_implicit_acc'].idxmax(), 'layer']),
                'best_mm_implicit_acc': float(feature_df['mm_implicit_acc'].max()),
                'best_wmm_explicit_layer': int(feature_df.loc[feature_df['wmm_explicit_test_acc'].idxmax(), 'layer']),
                'best_wmm_explicit_acc': float(feature_df['wmm_explicit_test_acc'].max()),
                'best_wmm_implicit_layer': int(feature_df.loc[feature_df['wmm_implicit_acc'].idxmax(), 'layer']),
                'best_wmm_implicit_acc': float(feature_df['wmm_implicit_acc'].max()),
            })
    return pd.DataFrame(summary_rows)


STYLE_LAST_ONLY = {
    'lr': {'label': 'LR', 'color': 'C0', 'marker': 'o'},
    'wlr': {'label': 'WLR', 'color': 'C1', 'marker': 'D'},
    'mm': {'label': 'MM', 'color': 'C2', 'marker': 's'},
    'wmm': {'label': 'WMM', 'color': 'C3', 'marker': '^'},
}
STYLE_MEAN_ONLY = STYLE_LAST_ONLY
STYLE_COMPARISON = {
    'lr': {
        'last': {'color': 'C0', 'marker': 'o', 'linestyle': '-', 'label': 'LR, last answer token'},
        'mean': {'color': 'C0', 'marker': 'o', 'linestyle': '--', 'label': 'LR, mean answer tokens'},
    },
    'wlr': {
        'last': {'color': 'C1', 'marker': 'D', 'linestyle': '-', 'label': 'WLR, last answer token'},
        'mean': {'color': 'C1', 'marker': 'D', 'linestyle': '--', 'label': 'WLR, mean answer tokens'},
    },
    'mm': {
        'last': {'color': 'C2', 'marker': 's', 'linestyle': '-', 'label': 'MM, last answer token'},
        'mean': {'color': 'C2', 'marker': 's', 'linestyle': '--', 'label': 'MM, mean answer tokens'},
    },
    'wmm': {
        'last': {'color': 'C3', 'marker': '^', 'linestyle': '-', 'label': 'WMM, last answer token'},
        'mean': {'color': 'C3', 'marker': '^', 'linestyle': '--', 'label': 'WMM, mean answer tokens'},
    },
}


def add_line_only_legend(ax, *, loc='lower left', fontsize=8, ncol=1):
    from matplotlib.lines import Line2D
    handles, labels = ax.get_legend_handles_labels()
    line_only_handles = [
        Line2D([0], [0], color=handle.get_color(), linestyle=handle.get_linestyle(), linewidth=handle.get_linewidth())
        for handle in handles
    ]
    ax.legend(line_only_handles, labels, loc=loc, fontsize=fontsize, ncol=ncol)


def draw_pooling_only_figure(df, title_prefix, style_map):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)
    for ax, metric, title in [
        (axes[0], 'explicit_test_acc', 'Explicit test accuracy by layer'),
        (axes[1], 'implicit_acc', 'Implicit accuracy by layer'),
    ]:
        for family in ['lr', 'wlr', 'mm', 'wmm']:
            style = style_map[family]
            ax.plot(
                df['layer'],
                df[f'{family}_{metric}'],
                marker=style['marker'],
                linewidth=2,
                color=style['color'],
                label=style['label'],
            )
        ax.set_title(title)
        ax.set_xlabel('Layer')
        ax.set_ylabel('Accuracy')
        ax.set_ylim(0.45, 1.02)
        ax.grid(True, alpha=0.3)
        add_line_only_legend(ax, loc='lower left', fontsize=8)
    fig.suptitle(title_prefix, y=1.03)
    return fig


def draw_comparison_figure(last_df, mean_df, title_prefix):
    fig, axes = plt.subplots(1, 2, figsize=(15, 5.5), constrained_layout=True)
    for ax, metric, title in [
        (axes[0], 'explicit_test_acc', 'Explicit test accuracy by layer'),
        (axes[1], 'implicit_acc', 'Implicit accuracy by layer'),
    ]:
        for family in ['lr', 'wlr', 'mm', 'wmm']:
            last_style = STYLE_COMPARISON[family]['last']
            mean_style = STYLE_COMPARISON[family]['mean']
            ax.plot(last_df['layer'], last_df[f'{family}_{metric}'], linewidth=2, marker=last_style['marker'], linestyle=last_style['linestyle'], color=last_style['color'], label=last_style['label'])
            ax.plot(mean_df['layer'], mean_df[f'{family}_{metric}'], linewidth=2, marker=mean_style['marker'], linestyle=mean_style['linestyle'], color=mean_style['color'], label=mean_style['label'])
        ax.set_title(title)
        ax.set_xlabel('Layer')
        ax.set_ylabel('Accuracy')
        ax.set_ylim(0.45, 1.02)
        ax.grid(True, alpha=0.3)
        add_line_only_legend(ax, loc='lower left', fontsize=8, ncol=2)
    fig.suptitle(title_prefix, y=1.03)
    return fig


def draw_mean_training_comparison_three_way_figure(explicit_only_df, one_run_df, two_run_df, title_prefix):
    family_styles = {
        'lr': {'color': 'C0', 'label': 'LR'},
        'wlr': {'color': 'C1', 'label': 'WLR'},
        'mm': {'color': 'C2', 'label': 'MM'},
        'wmm': {'color': 'C3', 'label': 'WMM'},
    }
    training_styles = [
        (explicit_only_df, '-', 'explicit train only'),
        (one_run_df, '--', '+ March 22, 2026 red-team'),
        (two_run_df, ':', '+ March 22 + April 3, 2026 red-team'),
    ]
    fig, axes = plt.subplots(1, 2, figsize=(15, 5.5), constrained_layout=True)
    for ax, metric, title in [
        (axes[0], 'explicit_test_acc', 'Explicit test accuracy by layer'),
        (axes[1], 'implicit_acc', 'Implicit accuracy by layer'),
    ]:
        for family, style in family_styles.items():
            for df, linestyle, suffix in training_styles:
                ax.plot(df['layer'], df[f'{family}_{metric}'], color=style['color'], linestyle=linestyle, linewidth=2, marker='o', label=f"{style['label']}, {suffix}")
        ax.set_title(title)
        ax.set_xlabel('Layer')
        ax.set_ylabel('Accuracy')
        ax.set_ylim(0.45, 1.02)
        ax.grid(True, alpha=0.3)
        add_line_only_legend(ax, loc='lower left', fontsize=8, ncol=2)
    fig.suptitle(title_prefix, y=1.03)
    return fig


def fit_probe_regime(feature_acts_exp, feature_acts_imp, y_exp, y_imp, *, train_idx, test_idx, train_dataset_name, train_dataset_label, X_extra_train=None, y_extra_train=None):
    rows = []
    artifact_rows = []
    for layer in SELECTED_LAYERS:
        X_train = feature_acts_exp[layer][train_idx]
        y_train = y_exp[train_idx]
        if X_extra_train is not None and y_extra_train is not None:
            X_train = np.concatenate([X_train, X_extra_train[layer]], axis=0)
            y_train = np.concatenate([y_train, y_extra_train], axis=0)
        X_test = feature_acts_exp[layer][test_idx]
        y_test = y_exp[test_idx]
        X_imp = feature_acts_imp[layer]

        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            lr_probe = LogisticRegression(max_iter=1000, random_state=PAIR_SPLIT_RANDOM_STATE)
            lr_probe.fit(X_train, y_train)

        whitener = fit_whitener(X_train, reg=WHITEN_REG)
        X_train_w = apply_whitener(X_train, whitener)
        X_test_w = apply_whitener(X_test, whitener)
        X_imp_w = apply_whitener(X_imp, whitener)

        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            wlr_probe = LogisticRegression(max_iter=1000, random_state=PAIR_SPLIT_RANDOM_STATE)
            wlr_probe.fit(X_train_w, y_train)

        mm_direction = train_mm_probe(X_train, y_train)
        mm_pred_train, _ = mm_predict(X_train, mm_direction)
        mm_pred_test, _ = mm_predict(X_test, mm_direction)
        mm_pred_imp, _ = mm_predict(X_imp, mm_direction)
        mm_steering_vector, mm_raw_norm = normalize_direction(mm_direction) if NORMALIZE_STEERING_VECTORS else (mm_direction.astype(np.float32), float(np.linalg.norm(mm_direction)))

        wmm_model = train_whitened_mm_probe(X_train, y_train, reg=WHITEN_REG)
        wmm_pred_train, _ = whitened_mm_predict(X_train, wmm_model)
        wmm_pred_test, _ = whitened_mm_predict(X_test, wmm_model)
        wmm_pred_imp, _ = whitened_mm_predict(X_imp, wmm_model)
        wmm_steering_vector, wmm_raw_norm = normalize_direction(wmm_model['effective_direction']) if NORMALIZE_STEERING_VECTORS else (wmm_model['effective_direction'].astype(np.float32), float(np.linalg.norm(wmm_model['effective_direction'])))

        lr_coef = lr_probe.coef_[0].astype(np.float32)
        lr_intercept = float(lr_probe.intercept_[0])

        wlr_whitened_coef = wlr_probe.coef_[0].astype(np.float32)
        wlr_effective_coef = (whitener['inv_sqrt'] @ wlr_whitened_coef).astype(np.float32)
        wlr_effective_intercept = float(wlr_probe.intercept_[0] - np.dot(whitener['mean_train'], wlr_effective_coef))
        wlr_train_pred = ((X_train @ wlr_effective_coef) + wlr_effective_intercept > 0).astype(np.int64)
        wlr_test_pred = ((X_test @ wlr_effective_coef) + wlr_effective_intercept > 0).astype(np.int64)
        wlr_imp_pred = ((X_imp @ wlr_effective_coef) + wlr_effective_intercept > 0).astype(np.int64)

        rows.append({
            'train_dataset': train_dataset_name,
            'train_dataset_label': train_dataset_label,
            'layer': int(layer),
            'base_train_size': int(len(train_idx)),
            'extra_train_size': 0 if y_extra_train is None else int(len(y_extra_train)),
            'train_size': int(len(y_train)),
            'lr_train_acc': float((lr_probe.predict(X_train) == y_train).mean()),
            'lr_explicit_test_acc': float((lr_probe.predict(X_test) == y_test).mean()),
            'lr_implicit_acc': float((lr_probe.predict(X_imp) == y_imp).mean()),
            'wlr_train_acc': float((wlr_train_pred == y_train).mean()),
            'wlr_explicit_test_acc': float((wlr_test_pred == y_test).mean()),
            'wlr_implicit_acc': float((wlr_imp_pred == y_imp).mean()),
            'mm_train_acc': float((mm_pred_train == y_train).mean()),
            'mm_explicit_test_acc': float((mm_pred_test == y_test).mean()),
            'mm_implicit_acc': float((mm_pred_imp == y_imp).mean()),
            'wmm_train_acc': float((wmm_pred_train == y_train).mean()),
            'wmm_explicit_test_acc': float((wmm_pred_test == y_test).mean()),
            'wmm_implicit_acc': float((wmm_pred_imp == y_imp).mean()),
            'whitener_cov_reg_condition_number': float(whitener['cov_reg_condition_number']),
            'wmm_reg': float(wmm_model['reg']),
            'mm_raw_norm': float(mm_raw_norm),
            'mm_steering_norm': float(np.linalg.norm(mm_steering_vector)),
            'wmm_raw_norm': float(wmm_raw_norm),
            'wmm_steering_norm': float(np.linalg.norm(wmm_steering_vector)),
        })

        artifact_rows.append({
            'layer': int(layer),
            'lr_coef': lr_coef,
            'lr_intercept': np.float32(lr_intercept),
            'wlr_effective_coef': wlr_effective_coef,
            'wlr_effective_intercept': np.float32(wlr_effective_intercept),
            'mm_raw_direction': mm_direction.astype(np.float32),
            'mm_steering_vector': mm_steering_vector.astype(np.float32),
            'wmm_effective_direction': wmm_model['effective_direction'].astype(np.float32),
            'wmm_steering_vector': wmm_steering_vector.astype(np.float32),
            'wmm_mean_train': wmm_model['mean_train'].astype(np.float32),
        })

        clear_gpu_cache()

    rows_df = pd.DataFrame(rows).sort_values('layer').reset_index(drop=True)
    artifact_rows = sorted(artifact_rows, key=lambda row: row['layer'])
    return rows_df, artifact_rows


In [ ]:
explicit_pair_indices = np.arange(len(explicit_pairs), dtype=np.int64)
explicit_train_pair_idx, explicit_test_pair_idx = train_test_split(
    explicit_pair_indices,
    test_size=0.2,
    random_state=PAIR_SPLIT_RANDOM_STATE,
    shuffle=True,
)
explicit_train_pair_idx = np.sort(explicit_train_pair_idx.astype(np.int64))
explicit_test_pair_idx = np.sort(explicit_test_pair_idx.astype(np.int64))

train_idx = np.sort(np.concatenate([
    2 * explicit_train_pair_idx,
    2 * explicit_train_pair_idx + 1,
]).astype(np.int64))
test_idx = np.sort(np.concatenate([
    2 * explicit_test_pair_idx,
    2 * explicit_test_pair_idx + 1,
]).astype(np.int64))

if np.intersect1d(train_idx, test_idx).size != 0:
    raise ValueError('Train/test example indices overlap after pair-level split.')
if len(train_idx) + len(test_idx) != len(y_exp):
    raise ValueError('Pair-derived example split does not cover all explicit examples.')

print('Explicit pair split:', f'train pairs = {len(explicit_train_pair_idx)}', f'| test pairs = {len(explicit_test_pair_idx)}')
print('Explicit example split implied by pair split:', f'train examples = {len(train_idx)}', f'| test examples = {len(test_idx)}')
print('Extracting explicit/implicit Qwen activations...')
explicit_feature_acts = extract_answer_token_activations_qwen(explicit_examples, batch_size=PROBE_BATCH_SIZE, dataset_name='explicit_expanded')
implicit_feature_acts = extract_answer_token_activations_qwen(implicit_examples, batch_size=PROBE_BATCH_SIZE, dataset_name='implicit_expanded')
print(f'Explicit mean layer {SELECTED_LAYERS[0]} shape:', explicit_feature_acts['mean_answer_tokens'][SELECTED_LAYERS[0]].shape)
print(f'Implicit mean layer {SELECTED_LAYERS[0]} shape:', implicit_feature_acts['mean_answer_tokens'][SELECTED_LAYERS[0]].shape)
print(f'Explicit last layer {SELECTED_LAYERS[0]} shape:', explicit_feature_acts['last_answer_token'][SELECTED_LAYERS[0]].shape)
print(f'Implicit last layer {SELECTED_LAYERS[0]} shape:', implicit_feature_acts['last_answer_token'][SELECTED_LAYERS[0]].shape)

print('Extracting red-team activations for primary run...')
primary_red_team_feature_acts = extract_answer_token_activations_qwen(primary_red_team_examples, batch_size=PROBE_BATCH_SIZE, dataset_name=f'redteam_{PRIMARY_RED_TEAM_RUN_ID}')
print('Extracting red-team activations for latest run...')
latest_red_team_feature_acts = extract_answer_token_activations_qwen(latest_red_team_examples, batch_size=PROBE_BATCH_SIZE, dataset_name=f'redteam_{LATEST_RED_TEAM_RUN_ID}')

y_two_red_team_runs = np.concatenate([y_primary_red_team, y_latest_red_team], axis=0)
two_red_team_feature_acts = {
    feature_name: {
        layer: np.concatenate([
            primary_red_team_feature_acts[feature_name][layer],
            latest_red_team_feature_acts[feature_name][layer],
        ], axis=0).astype(np.float32)
        for layer in SELECTED_LAYERS
    }
    for feature_name in [feature_spec['name'] for feature_spec in FEATURE_SPECS]
}

TRAIN_REGIMES = [
    {
        'name': 'explicit_train_only',
        'label': 'explicit train only',
        'extra_feature_acts': None,
        'extra_labels': None,
    },
    {
        'name': f'explicit_train_plus_redteam_{PRIMARY_RED_TEAM_RUN_ID}',
        'label': '+ March 22, 2026 red-team',
        'extra_feature_acts': primary_red_team_feature_acts,
        'extra_labels': y_primary_red_team,
    },
    {
        'name': f'explicit_train_plus_redteam_{TWO_RED_TEAM_RUNS_SLUG}',
        'label': '+ March 22 + April 3, 2026 red-team',
        'extra_feature_acts': two_red_team_feature_acts,
        'extra_labels': y_two_red_team_runs,
    },
]

all_metric_parts = []
artifact_store = {}
regime_results = {}

for regime in TRAIN_REGIMES:
    regime_results[regime['name']] = {}
    artifact_store[regime['name']] = {}
    for feature_spec in FEATURE_SPECS:
        feature_name = feature_spec['name']
        print(f"Training {feature_name} probes for regime={regime['name']}")
        metrics_df, artifact_rows = fit_probe_regime(
            explicit_feature_acts[feature_name],
            implicit_feature_acts[feature_name],
            y_exp,
            y_imp,
            train_idx=train_idx,
            test_idx=test_idx,
            train_dataset_name=regime['name'],
            train_dataset_label=regime['label'],
            X_extra_train=None if regime['extra_feature_acts'] is None else regime['extra_feature_acts'][feature_name],
            y_extra_train=regime['extra_labels'],
        )
        metrics_df.insert(2, 'feature_name', feature_name)
        all_metric_parts.append(metrics_df)
        regime_results[regime['name']][feature_name] = metrics_df.copy()
        artifact_store[regime['name']][feature_name] = artifact_rows

all_metrics_df = pd.concat(all_metric_parts, ignore_index=True)
all_metrics_df = all_metrics_df.sort_values(['train_dataset', 'feature_name', 'layer']).reset_index(drop=True)
summary_df = summarize_probe_metrics(all_metrics_df)

display(all_metrics_df.head(20))
display(summary_df)


In [ ]:
train_regime_names = [regime['name'] for regime in TRAIN_REGIMES]
train_regime_labels = [regime['label'] for regime in TRAIN_REGIMES]
feature_names = [feature_spec['name'] for feature_spec in FEATURE_SPECS]
layer_array = np.asarray(SELECTED_LAYERS, dtype=np.int64)

artifact_shape = (len(train_regime_names), len(feature_names), len(SELECTED_LAYERS), hidden_size)
scalar_shape = (len(train_regime_names), len(feature_names), len(SELECTED_LAYERS))

lr_coef = np.zeros(artifact_shape, dtype=np.float32)
lr_intercept = np.zeros(scalar_shape, dtype=np.float32)
wlr_effective_coef = np.zeros(artifact_shape, dtype=np.float32)
wlr_effective_intercept = np.zeros(scalar_shape, dtype=np.float32)
mm_raw_directions = np.zeros(artifact_shape, dtype=np.float32)
mm_steering_vectors = np.zeros(artifact_shape, dtype=np.float32)
wmm_effective_directions = np.zeros(artifact_shape, dtype=np.float32)
wmm_steering_vectors = np.zeros(artifact_shape, dtype=np.float32)
wmm_mean_train = np.zeros(artifact_shape, dtype=np.float32)

for regime_idx, regime_name in enumerate(train_regime_names):
    for feature_idx, feature_name in enumerate(feature_names):
        artifact_rows = artifact_store[regime_name][feature_name]
        for layer_idx, layer in enumerate(SELECTED_LAYERS):
            row = artifact_rows[layer_idx]
            if row['layer'] != layer:
                raise ValueError(f'Artifact row order mismatch for {regime_name} / {feature_name}: expected layer {layer}, got {row["layer"]}')
            lr_coef[regime_idx, feature_idx, layer_idx, :] = row['lr_coef']
            lr_intercept[regime_idx, feature_idx, layer_idx] = row['lr_intercept']
            wlr_effective_coef[regime_idx, feature_idx, layer_idx, :] = row['wlr_effective_coef']
            wlr_effective_intercept[regime_idx, feature_idx, layer_idx] = row['wlr_effective_intercept']
            mm_raw_directions[regime_idx, feature_idx, layer_idx, :] = row['mm_raw_direction']
            mm_steering_vectors[regime_idx, feature_idx, layer_idx, :] = row['mm_steering_vector']
            wmm_effective_directions[regime_idx, feature_idx, layer_idx, :] = row['wmm_effective_direction']
            wmm_steering_vectors[regime_idx, feature_idx, layer_idx, :] = row['wmm_steering_vector']
            wmm_mean_train[regime_idx, feature_idx, layer_idx, :] = row['wmm_mean_train']

metrics_path = SAVE_DIR / f'qwen3_4b_question_options_answer_probe_metrics_{RUN_ID}.csv'
summary_path = SAVE_DIR / f'qwen3_4b_question_options_answer_probe_summary_{RUN_ID}.csv'
artifact_path = SAVE_DIR / f'qwen3_4b_question_options_answer_probe_artifacts_{RUN_ID}.npz'
metadata_path = SAVE_DIR / f'qwen3_4b_question_options_answer_probe_metadata_{RUN_ID}.json'
figure_index_path = SAVE_DIR / f'qwen3_4b_question_options_answer_probe_figures_{RUN_ID}.csv'

all_metrics_df.to_csv(metrics_path, index=False)
summary_df.to_csv(summary_path, index=False)

np.savez_compressed(
    artifact_path,
    train_regimes=np.asarray(train_regime_names),
    train_regime_labels=np.asarray(train_regime_labels),
    feature_names=np.asarray(feature_names),
    layers=layer_array,
    explicit_train_pair_indices=np.asarray(explicit_train_pair_idx, dtype=np.int64),
    explicit_test_pair_indices=np.asarray(explicit_test_pair_idx, dtype=np.int64),
    explicit_train_example_indices=np.asarray(train_idx, dtype=np.int64),
    explicit_test_example_indices=np.asarray(test_idx, dtype=np.int64),
    lr_coef=lr_coef,
    lr_intercept=lr_intercept,
    wlr_effective_coef=wlr_effective_coef,
    wlr_effective_intercept=wlr_effective_intercept,
    mm_raw_directions=mm_raw_directions,
    mm_steering_vectors=mm_steering_vectors,
    wmm_effective_directions=wmm_effective_directions,
    wmm_steering_vectors=wmm_steering_vectors,
    wmm_mean_train=wmm_mean_train,
)

metadata_payload = {
    'run_id': RUN_ID,
    'model_name': MODEL_NAME,
    'selected_layers': SELECTED_LAYERS,
    'feature_names': feature_names,
    'train_regimes': train_regime_names,
    'train_regime_labels': train_regime_labels,
    'explicit_split_random_state': PAIR_SPLIT_RANDOM_STATE,
    'explicit_split_granularity': 'pair',
    'explicit_split_strategy': 'pair_level_80_20',
    'explicit_train_pair_count': int(len(explicit_train_pair_idx)),
    'explicit_test_pair_count': int(len(explicit_test_pair_idx)),
    'explicit_train_example_count': int(len(train_idx)),
    'explicit_test_example_count': int(len(test_idx)),
    'strip_option_letters_for_probe_training': bool(STRIP_OPTION_LETTERS_FOR_PROBE_TRAINING),
    'use_chat_template': bool(USE_CHAT_TEMPLATE),
    'probe_prompt_use_chat_template': bool(USE_CHAT_TEMPLATE),
    'disable_thinking_trace': bool(DISABLE_THINKING_TRACE),
    'probe_prompt_disable_thinking_trace': bool(DISABLE_THINKING_TRACE),
    'normalize_steering_vectors': bool(NORMALIZE_STEERING_VECTORS),
    'patch_prompt_last_only': bool(PATCH_PROMPT_LAST_ONLY),
    'patch_generation_tokens': bool(PATCH_GENERATION_TOKENS),
    'whiten_reg': float(WHITEN_REG),
    'probe_batch_size': int(PROBE_BATCH_SIZE),
    'explicit_expanded_path': str(explicit_expanded_path),
    'implicit_expanded_path': str(implicit_expanded_path),
    'explicit_expanded_sha256': sha256(explicit_expanded_path),
    'implicit_expanded_sha256': sha256(implicit_expanded_path),
    'primary_red_team_run_id': PRIMARY_RED_TEAM_RUN_ID,
    'latest_red_team_run_id': LATEST_RED_TEAM_RUN_ID,
    'two_red_team_runs_slug': TWO_RED_TEAM_RUNS_SLUG,
    'primary_red_team_path': str(primary_red_team_path),
    'latest_red_team_path': str(latest_red_team_path),
    'recommended_time_utility_train_regime': 'explicit_train_only',
    'recommended_time_utility_feature_name': 'mean_answer_tokens',
    'recommended_time_utility_vector_key': 'mm_steering_vectors',
    'recommended_time_utility_layer': 22,
    'artifact_format_version': 4,
}
metadata_path.write_text(json.dumps(metadata_payload, indent=2) + '\n', encoding='utf-8')

figure_rows = []
for regime_name in train_regime_names:
    safe_regime_name = regime_name.replace('/', '_')
    last_df = regime_results[regime_name]['last_answer_token']
    mean_df = regime_results[regime_name]['mean_answer_tokens']
    last_fig = draw_pooling_only_figure(last_df, f'{MODEL_NAME}: {regime_name} | last answer-token probes', STYLE_LAST_ONLY)
    mean_fig = draw_pooling_only_figure(mean_df, f'{MODEL_NAME}: {regime_name} | mean answer-token probes', STYLE_MEAN_ONLY)
    comparison_fig = draw_comparison_figure(last_df, mean_df, f'{MODEL_NAME}: {regime_name} | pooling-method comparison')

    last_fig_path = SAVE_DIR / f'{safe_regime_name}_last_answer_token_{RUN_ID}.png'
    mean_fig_path = SAVE_DIR / f'{safe_regime_name}_mean_answer_tokens_{RUN_ID}.png'
    comparison_fig_path = SAVE_DIR / f'{safe_regime_name}_comparison_{RUN_ID}.png'

    last_fig.savefig(last_fig_path, dpi=200, bbox_inches='tight')
    mean_fig.savefig(mean_fig_path, dpi=200, bbox_inches='tight')
    comparison_fig.savefig(comparison_fig_path, dpi=200, bbox_inches='tight')
    plt.close(last_fig)
    plt.close(mean_fig)
    plt.close(comparison_fig)

    figure_rows.extend([
        {'train_dataset': regime_name, 'figure_type': 'last_answer_token', 'path': str(last_fig_path)},
        {'train_dataset': regime_name, 'figure_type': 'mean_answer_tokens', 'path': str(mean_fig_path)},
        {'train_dataset': regime_name, 'figure_type': 'comparison', 'path': str(comparison_fig_path)},
    ])

three_way_fig = draw_mean_training_comparison_three_way_figure(
    regime_results['explicit_train_only']['mean_answer_tokens'],
    regime_results[f'explicit_train_plus_redteam_{PRIMARY_RED_TEAM_RUN_ID}']['mean_answer_tokens'],
    regime_results[f'explicit_train_plus_redteam_{TWO_RED_TEAM_RUNS_SLUG}']['mean_answer_tokens'],
    f'{MODEL_NAME}: mean answer-token probes | explicit-only vs one-run vs two-run red-team augmentation',
)
three_way_fig_path = SAVE_DIR / f'mean_training_comparison_{RUN_ID}.png'
three_way_fig.savefig(three_way_fig_path, dpi=200, bbox_inches='tight')
plt.close(three_way_fig)
figure_rows.append({'train_dataset': 'all', 'figure_type': 'mean_training_comparison', 'path': str(three_way_fig_path)})

figure_df = pd.DataFrame(figure_rows)
figure_df.to_csv(figure_index_path, index=False)
display(figure_df)

print('Saved metrics  :', metrics_path)
print('Saved summary  :', summary_path)
print('Saved artifacts:', artifact_path)
print('Saved metadata :', metadata_path)
print('Saved figures  :', figure_index_path)
print('Saved figures in:', SAVE_DIR)
